In [ ]:
# import torch
# import torchvision
# import torchvision.transforms as transforms
# from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
# import torch.nn as nn
# import torch.optim as optim
# from torch.cuda.amp import GradScaler, autocast
# import wandb
# import os
# import time

# os.environ["WANDB_MODE"] = "DISABLED"

# try:
#     wandb.finish()  # ✅ cleanly close any previous run
# except:
#     pass

# time.sleep(3)  # ✅ ensure previous run is fully closed


In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=256, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)


class VATSA_VisualEncoder(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=10, freeze_backbone=True):
        super().__init__()
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        self.backbone = efficientnet_b0(weights=weights)
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.projection = nn.Linear(num_features, embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.projection(features)
        logits = self.classifier(embedding)
        return logits, embedding


device = "cuda" if torch.cuda.is_available() else "cpu"
model = VATSA_VisualEncoder(embedding_dim=512, num_classes=10, freeze_backbone=True).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

for epoch in range(20):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            logits, _ = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

    scheduler.step()
    avg_loss = running_loss / len(train_loader)

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

    acc = correct / total * 100
    print(f"Epoch {epoch+1}/20 - Loss: {avg_loss:.4f} - Test Acc: {acc:.2f}%")

torch.save({
    'model_state': model.state_dict(),
    'embedding_dim': 512
}, "vatsa_visual_encoder_cifar10.pth")

print("✅ VATSA Visual Encoder saved!")

c:\Users\vinay\OneDrive\Desktop\VATSA\VATSA\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Epoch 1/20 - Loss: 0.9546 - Test Acc: 75.59%
Epoch 2/20 - Loss: 0.8128 - Test Acc: 76.37%
Epoch 3/20 - Loss: 0.7955 - Test Acc: 77.62%
Epoch 4/20 - Loss: 0.7780 - Test Acc: 76.86%
Epoch 5/20 - Loss: 0.7742 - Test Acc: 76.35%
Epoch 6/20 - Loss: 0.7649 - Test Acc: 77.37%
Epoch 7/20 - Loss: 0.7591 - Test Acc: 77.33%
Epoch 8/20 - Loss: 0.7471 - Test Acc: 78.65%
Epoch 9/20 - Loss: 0.7490 - Test Acc: 78.12%
Epoch 10/20 - Loss: 0.7438 - Test Acc: 77.68%
Epoch 11/20 - Loss: 0.7406 - Test Acc: 78.00%
Epoch 12/20 - Loss: 0.7320 - Test Acc: 78.70%
Epoch 13/20 - Loss: 0.7266 - Test Acc: 78.31%
Epoch 14/20 - Loss: 0.7220 - Test Acc: 78.43%
Epoch 15/20 - Loss: 0.7215 - Test Acc: 78.88%
Epoch 16/20 - Loss: 0.7199 - Test Acc: 79.04%
Epoch 17/20 - Loss: 0.7171 - Test Acc: 79.04%
Epoch 18/20 - Loss: 0.7023 - Test Acc: 79.16%
Epoch 19/20 - Loss: 0.7078 - Test Acc: 79.13%
Epoch 20/20 - Loss: 0.7069 - Test Acc: 79.19%
✅ VATSA Visual Encoder saved!


In [17]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn

class VATSA_VisualEncoder(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=10, freeze_backbone=True):
        super().__init__()
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        self.backbone = efficientnet_b0(weights=weights)
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.projection = nn.Linear(num_features, embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)  # ✅ matches training code

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.projection(features)
        logits = self.classifier(embedding)
        return logits, embedding

device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint = torch.load("vatsa_visual_encoder_cifar10.pth", map_location=device)
model = VATSA_VisualEncoder(embedding_dim=512, num_classes=10).to(device)
model.load_state_dict(checkpoint['model_state'])
model.eval()

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)

correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        logits, _ = model(images)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {correct/total*100:.2f}%")

Test Accuracy: 79.19%


In [20]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=256, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)


class VATSA_VisualEncoder(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=10, freeze_backbone=True):
        super().__init__()
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        self.backbone = efficientnet_b0(weights=weights)
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.projection = nn.Linear(num_features, embedding_dim)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),                          # ✅ dropout added
            nn.Linear(embedding_dim, num_classes)
        )

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.projection(features)
        logits = self.classifier(embedding)
        return logits, embedding


device = "cuda" if torch.cuda.is_available() else "cpu"

# ✅ Load previous checkpoint and continue training
checkpoint = torch.load("vatsa_visual_encoder_cifar10.pth", map_location=device)
model = VATSA_VisualEncoder(embedding_dim=512, num_classes=10, freeze_backbone=True).to(device)
model.load_state_dict(checkpoint['model_state'], strict=False)  # strict=False handles classifier shape change

# ✅ Unfreeze last few backbone layers for fine-tuning
for param in model.backbone.features[6:].parameters():
    param.requires_grad = True

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

best_acc = 0.0

for epoch in range(20):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            logits, _ = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

    scheduler.step()
    avg_loss = running_loss / len(train_loader)

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

    acc = correct / total * 100

    # ✅ Save best model only
    if acc > best_acc:
        best_acc = acc
        torch.save({
            'model_state': model.state_dict(),
            'embedding_dim': 512
        }, "vatsa_visual_encoder_cifar10.pth")
        print(f"Epoch {epoch+1}/20 - Loss: {avg_loss:.4f} - Test Acc: {acc:.2f}% ✅ Best saved")
    else:
        print(f"Epoch {epoch+1}/20 - Loss: {avg_loss:.4f} - Test Acc: {acc:.2f}%")

print(f"\nBest Accuracy: {best_acc:.2f}%")

c:\Users\vinay\OneDrive\Desktop\VATSA\VATSA\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Epoch 1/20 - Loss: 0.7434 - Test Acc: 89.06% ✅ Best saved
Epoch 2/20 - Loss: 0.3698 - Test Acc: 90.95% ✅ Best saved
Epoch 3/20 - Loss: 0.3012 - Test Acc: 91.67% ✅ Best saved
Epoch 4/20 - Loss: 0.2567 - Test Acc: 92.33% ✅ Best saved
Epoch 5/20 - Loss: 0.2305 - Test Acc: 92.80% ✅ Best saved
Epoch 6/20 - Loss: 0.2067 - Test Acc: 93.27% ✅ Best saved
Epoch 7/20 - Loss: 0.1881 - Test Acc: 93.22%
Epoch 8/20 - Loss: 0.1741 - Test Acc: 93.63% ✅ Best saved
Epoch 9/20 - Loss: 0.1623 - Test Acc: 93.69% ✅ Best saved
Epoch 10/20 - Loss: 0.1526 - Test Acc: 93.99% ✅ Best saved
Epoch 11/20 - Loss: 0.1433 - Test Acc: 93.98%
Epoch 12/20 - Loss: 0.1334 - Test Acc: 94.12% ✅ Best saved
Epoch 13/20 - Loss: 0.1280 - Test Acc: 94.21% ✅ Best saved
Epoch 14/20 - Loss: 0.1251 - Test Acc: 94.39% ✅ Best saved
Epoch 15/20 - Loss: 0.1210 - Test Acc: 94.23%
Epoch 16/20 - Loss: 0.1141 - Test Acc: 94.32%
Epoch 17/20 - Loss: 0.1127 - Test Acc: 94.30%
Epoch 18/20 - Loss: 0.1118 - Test Acc: 94.46% ✅ Best saved
Epoch 19/20 